# Notebook 04: Satellite Integration

This notebook demonstrates the **satellite remote sensing pipeline** from the
`ragweed_toolkit.satellite` module (Chapter 5, Act III -- Satellite Integration).

The pipeline connects Sentinel-2 spectral data with field-level weed density:

1. **Earth Engine export** -- NDVI composites and time series (shown as patterns)
2. **Spectral indices** -- 9 indices characterizing vegetation, soil, and moisture
3. **PRESTO embeddings** -- Foundation model embeddings (shown as pattern)
4. **Risk zone classification** -- Combining NDVI statistics into management zones
5. **Correlation analysis** -- Linking satellite features to weed observations

All data is synthetic. Earth Engine and PRESTO operations are shown as
code patterns (wrapped in `if False:` blocks) since they require authentication.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, box

np.random.seed(42)

## 1. Earth Engine Export Pattern

The toolkit wraps Earth Engine for exporting Sentinel-2 NDVI and RGB composites.
The code below shows the API pattern but does not execute (requires `ee.Authenticate()`).

In [ ]:
# === Earth Engine Export Pattern ===
# This block shows the API but does not execute.
# To run: pip install ragweed-ai-toolkit[satellite] && ee.Authenticate()

if False:
    import ee
    ee.Initialize()

    from ragweed_toolkit.satellite import (
        get_sentinel2_collection,
        mask_clouds_scl,
        export_ndvi_max,
        export_rgb_median,
        export_composites,
    )

    # Define a paddock boundary (WGS 84)
    paddock = ee.Geometry.Rectangle([-70.85, -34.20, -70.75, -34.10])

    # Export NDVI max and RGB median for the 2025-26 season
    export_composites(
        paddock,
        start_date="2025-10-01",
        end_date="2026-03-31",
        paddock_name="Santa_Ines",
        drive_folder="satellite_exports",
    )
    print("Export tasks submitted to Google Earth Engine")

print("Earth Engine export pattern shown above (not executed).")
print("See ragweed_toolkit.satellite.composites for full API.")

## 2. Spectral Indices from Sentinel-2

The toolkit computes 9 spectral indices from a 10-band Sentinel-2 stack.
Each index characterizes a different aspect of the surface:

| Index | Description |
|-------|-------------|
| NDVI | Vegetation density |
| EVI | Atmosphere-corrected vegetation |
| GNDVI | Chlorophyll content |
| RENDVI | Phenological state |
| S2WI | Soil moisture |
| NBR2 | Moisture and crop residue |
| BSI | Bare soil fraction |
| Clay | Clay minerals (SWIR ratio) |
| SWIRd | Surface texture proxy |

In [ ]:
from ragweed_toolkit.satellite.indices import (
    BAND_NAMES,
    INDEX_DESCRIPTIONS,
    compute_spectral_indices,
)

# Create a synthetic 10-band Sentinel-2 tile
H, W = 64, 64

band_ranges = {
    "B2":  (0.02, 0.10),  # Blue
    "B3":  (0.03, 0.12),  # Green
    "B4":  (0.02, 0.08),  # Red
    "B5":  (0.03, 0.15),  # Red Edge 1
    "B6":  (0.10, 0.30),  # Red Edge 2
    "B7":  (0.15, 0.40),  # Red Edge 3
    "B8":  (0.15, 0.50),  # NIR
    "B8A": (0.15, 0.45),  # Narrow NIR
    "B11": (0.05, 0.30),  # SWIR1
    "B12": (0.03, 0.20),  # SWIR2
}

spectral = np.zeros((10, H, W), dtype=np.float32)
for i, name in enumerate(BAND_NAMES):
    lo, hi = band_ranges[name]
    spectral[i] = np.random.uniform(lo, hi, (H, W))

print(f"=== Synthetic Sentinel-2 Image ===")
print(f"  Shape: {spectral.shape} (bands, H, W)")
print(f"  Bands: {', '.join(BAND_NAMES)}")

In [ ]:
# Compute all 9 indices
indices = compute_spectral_indices(spectral)

print(f"=== Computed Indices ({len(indices)} total) ===")
print(f"  {'Index':<8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}  Description")
print(f"  {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}  {'-'*40}")

for name, arr in indices.items():
    desc = INDEX_DESCRIPTIONS[name]
    print(f"  {name:<8} {arr.mean():>8.3f} {arr.std():>8.3f} "
          f"{arr.min():>8.3f} {arr.max():>8.3f}  {desc}")

In [ ]:
# Visualize 4 key indices
from ragweed_toolkit.viz.style import set_publication_style
set_publication_style()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, cmap) in zip(axes.flat, [
    ("NDVI", "RdYlGn"), ("BSI", "YlOrBr"),
    ("S2WI", "Blues"), ("Clay", "Oranges"),
]):
    im = ax.imshow(indices[name], cmap=cmap, aspect="equal")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"{name}: {INDEX_DESCRIPTIONS[name]}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Sentinel-2 Spectral Indices (Synthetic)", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 3. PRESTO Embedding Pattern

PRESTO is a satellite foundation model that compresses 12 months of Sentinel-2
data into 128-D embeddings. The code below shows the API pattern.

In [ ]:
# === PRESTO Embedding Pattern ===
# This block shows the API but does not execute.
# Requires: pip install presto-worldcereal

if False:
    from ragweed_toolkit.satellite import (
        extract_presto_embeddings,
        apply_clustering,
        apply_pca,
    )

    # Extract PRESTO embeddings for a paddock
    gdf_embeddings = extract_presto_embeddings(
        paddock_gdf=paddock_boundary,
        season="25-26",
        grid_size=10,  # 10m matching Sentinel-2 resolution
    )
    # gdf_embeddings has 128 embedding columns + geometry

    # Apply PCA for dimensionality reduction
    gdf_pca = apply_pca(gdf_embeddings, n_components=3)
    # adds PC1, PC2, PC3 columns

    # Cluster into management zones
    gdf_clustered = apply_clustering(gdf_pca, n_clusters=5)
    # adds 'cluster' column

print("PRESTO embedding pattern shown above (not executed).")
print("In the chapter, PRESTO PC1 correlated r=0.739 with AMBEL density.")

## 4. Risk Zone Classification

The risk zone classifier combines NDVI statistics into a weighted score
and assigns pixels to Low / Medium / High risk zones. This uses synthetic
NDVI data to demonstrate the workflow.

In [ ]:
from ragweed_toolkit.satellite.ndvi import create_risk_zones

# Generate synthetic NDVI time series statistics for a 10m grid
n_cells = 500
cell_x = np.random.uniform(0, 500, n_cells)
cell_y = np.random.uniform(0, 500, n_cells)

# Create spatial gradient (higher NDVI in healthy areas, lower near infestation)
dist_from_infestation = np.sqrt((cell_x - 300)**2 + (cell_y - 200)**2)
infestation_effect = np.exp(-dist_from_infestation / 150)

gdf_ndvi = gpd.GeoDataFrame(
    {
        "ndvi_max": 0.8 - 0.3 * infestation_effect + np.random.randn(n_cells) * 0.05,
        "ndvi_std": 0.05 + 0.15 * infestation_effect + np.random.randn(n_cells) * 0.02,
        "ndvi_slope": 0.02 - 0.05 * infestation_effect + np.random.randn(n_cells) * 0.01,
        "ndvi_decline": 0.1 + 0.3 * infestation_effect + np.random.randn(n_cells) * 0.05,
    },
    geometry=[Point(xi, yi) for xi, yi in zip(cell_x, cell_y)],
    crs="EPSG:32719",
)

# Classify risk zones
gdf_risk = create_risk_zones(gdf_ndvi)

print("=== Risk Zone Classification ===")
print(f"  Total cells: {len(gdf_risk)}")
print(f"  Risk score range: [{gdf_risk['risk_score'].min():.3f}, {gdf_risk['risk_score'].max():.3f}]")
print(f"\n  Zone distribution:")
print(gdf_risk["risk_zone"].value_counts().to_string())

In [ ]:
# Plot risk zones
risk_colors = {"Low": "#2ecc71", "Medium": "#f1c40f", "High": "#e74c3c"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Risk score (continuous)
sc = axes[0].scatter(
    [p.x for p in gdf_risk.geometry],
    [p.y for p in gdf_risk.geometry],
    c=gdf_risk["risk_score"], cmap="RdYlGn_r", s=20, alpha=0.8,
)
plt.colorbar(sc, ax=axes[0], label="Risk Score", shrink=0.8)
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
axes[0].set_title("(a) Continuous Risk Score", fontweight="bold")
axes[0].grid(True, alpha=0.2)

# Panel B: Risk zones (categorical)
for zone in ["Low", "Medium", "High"]:
    mask = gdf_risk["risk_zone"] == zone
    subset = gdf_risk[mask]
    axes[1].scatter(
        [p.x for p in subset.geometry],
        [p.y for p in subset.geometry],
        c=risk_colors[zone], s=20, alpha=0.8,
        label=f"{zone} (n={mask.sum()})",
        edgecolors="black", linewidths=0.2,
    )

axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Northing (m)")
axes[1].set_title("(b) Risk Zones", fontweight="bold")
axes[1].legend(title="Zone")
axes[1].grid(True, alpha=0.2)

plt.suptitle("Orobanche Risk Assessment from NDVI Statistics", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Satellite-Weed Correlation Analysis

Synthetic field observations linking satellite features to ground-truth weed
density. In the chapter, single-date NDVI achieved r = 0.837 (AMBEL) and
r = 0.890 (LENCU).

In [ ]:
# Generate correlated satellite-weed data
n_obs = 200
ndvi = np.random.uniform(0.3, 0.9, n_obs)

# AMBEL density: negatively correlated with NDVI (weeds depress vegetation)
ambel_density = np.maximum(0, 100 * (1 - ndvi) + np.random.randn(n_obs) * 10)

# LENCU density: also negatively correlated
lencu_density = np.maximum(0, 80 * (1 - ndvi) + np.random.randn(n_obs) * 8)

# Compute correlations
r_ambel = np.corrcoef(ndvi, ambel_density)[0, 1]
r_lencu = np.corrcoef(ndvi, lencu_density)[0, 1]

print("=== Satellite-Weed Correlations ===")
print(f"  NDVI vs AMBEL density: r = {r_ambel:.3f}")
print(f"  NDVI vs LENCU density: r = {r_lencu:.3f}")
print(f"\n  Chapter reference values:")
print(f"    NDVI-AMBEL: r = 0.837 (single-date Sept 2024)")
print(f"    NDVI-LENCU: r = 0.890 (single-date Sept 2024)")

In [ ]:
from ragweed_toolkit.viz.style import SPECIES_COLORS

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# AMBEL
axes[0].scatter(ndvi, ambel_density, c=SPECIES_COLORS["AMBEL"], s=20, alpha=0.6)
z = np.polyfit(ndvi, ambel_density, 1)
x_line = np.linspace(0.3, 0.9, 100)
axes[0].plot(x_line, np.polyval(z, x_line), "k--", linewidth=1.5)
axes[0].set_xlabel("NDVI")
axes[0].set_ylabel("AMBEL Density")
axes[0].set_title(f"(a) NDVI vs AMBEL (r = {r_ambel:.3f})", fontweight="bold")
axes[0].grid(True, alpha=0.2)

# LENCU
axes[1].scatter(ndvi, lencu_density, c=SPECIES_COLORS["LENCU"], s=20, alpha=0.6)
z = np.polyfit(ndvi, lencu_density, 1)
axes[1].plot(x_line, np.polyval(z, x_line), "k--", linewidth=1.5)
axes[1].set_xlabel("NDVI")
axes[1].set_ylabel("LENCU Density")
axes[1].set_title(f"(b) NDVI vs LENCU (r = {r_lencu:.3f})", fontweight="bold")
axes[1].grid(True, alpha=0.2)

plt.suptitle("Satellite-Weed Density Correlations", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the satellite integration pipeline:

1. **Earth Engine export** wraps Sentinel-2 cloud masking and composite generation
   for the crop season (Oct--Mar).

2. **9 spectral indices** characterize vegetation (NDVI, EVI, GNDVI), soil (BSI, Clay),
   and moisture (S2WI, NBR2, SWIRd) at each 10m pixel.

3. **PRESTO embeddings** compress 12 months of Sentinel-2 into 128-D vectors for
   zone clustering and PCA analysis.

4. **Risk zones** combine NDVI statistics (max, std, slope, decline) into a weighted
   score classified as Low / Medium / High.

5. Strong **negative correlations** (r = 0.84--0.89) between NDVI and weed density
   support satellite-based monitoring for early warning.

### Multi-Scale Integration

The key message from the chapter: no single scale solves the problem.

```
SATELLITE (10m)  -->  DRONE (1-5cm)  -->  DETECTION (mm)  -->  ADVISORY
  zone delineation     orthomosaic        species ID           management
  NDVI, PRESTO         ODM                YOLO + SAHI          risk zones
```

Together, these tools form a decision-support toolkit for weed management
under climate variability.